In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Thêm thư mục gốc vào path để import module
sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_and_filter_data, validate_extracted_text

# Thiết lập style cho biểu đồ
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Tải dữ liệu JD (Job Descriptions)
print("=" * 60)
print("ĐANG TẢI DỮ LIỆU JOB DESCRIPTIONS (JD)")
print("=" * 60)

df_jd = load_and_filter_data(
    csv_path='../data/raw/job_postings.csv',
    date_column='post_date',
    start_year=2021,
    end_year=2026
)

print(f"\nKích thước dữ liệu JD: {df_jd.shape}")
df_jd.head()

In [ ]:
# Tải dữ liệu CV (Resumes)
print("=" * 60)
print("ĐANG TẢI DỮ LIỆU RESUMES (CV)")
print("=" * 60)

df_cv = load_and_filter_data(
    csv_path='../data/raw/resumes.csv',
    date_column='last_update',
    start_year=2021,
    end_year=2026
)

print(f"\nKích thước dữ liệu CV: {df_cv.shape}")
df_cv.head()

In [ ]:
# Phân tích độ dài văn bản (số từ)
def count_words(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

df_jd['word_count'] = df_jd['description'].apply(count_words)
df_cv['word_count'] = df_cv['description'].apply(count_words)

print("=" * 60)
print("PHÂN TÍCH ĐỘ DÀI VĂN BẢN")
print("=" * 60)

print(f"\nJD - Số từ trung bình: {df_jd['word_count'].mean():.0f}")
print(f"JD - Median: {df_jd['word_count'].median():.0f}")
print(f"JD - Min: {df_jd['word_count'].min()}, Max: {df_jd['word_count'].max()}")

print(f"\nCV - Số từ trung bình: {df_cv['word_count'].mean():.0f}")
print(f"CV - Median: {df_cv['word_count'].median():.0f}")
print(f"CV - Min: {df_cv['word_count'].min()}, Max: {df_cv['word_count'].max()}")

print(f"\n=> JD dài gấp {df_jd['word_count'].mean() / df_cv['word_count'].mean():.1f} lần CV")

In [ ]:
# Vẽ biểu đồ phân bố độ dài văn bản
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# JD
axes[0].hist(df_jd['word_count'], bins=50, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Số từ', fontsize=12)
axes[0].set_ylabel('Tần suất', fontsize=12)
axes[0].set_title('Phân bố độ dài Job Descriptions', fontsize=14, fontweight='bold')
axes[0].axvline(df_jd['word_count'].mean(), color='red', linestyle='--', 
                label=f'TB: {df_jd["word_count"].mean():.0f} từ')
axes[0].legend()

# CV
axes[1].hist(df_cv['word_count'], bins=50, color='lightgreen', edgecolor='black')
axes[1].set_xlabel('Số từ', fontsize=12)
axes[1].set_ylabel('Tần suất', fontsize=12)
axes[1].set_title('Phân bố độ dài Resumes/CV', fontsize=14, fontweight='bold')
axes[1].axvline(df_cv['word_count'].mean(), color='red', linestyle='--', 
                label=f'TB: {df_cv["word_count"].mean():.0f} từ')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/eda_word_count_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Đã lưu biểu đồ tại: outputs/eda_word_count_distribution.png")

In [ ]:
# Phân tích xu hướng kỹ năng theo năm (nếu có cột category/skills)
print("=" * 60)
print("PHÂN TÍCH XU HƯỚNG THEO NĂM")
print("=" * 60)

# Giả sử JD có cột 'post_date' và 'category'
if 'post_date' in df_jd.columns and 'category' in df_jd.columns:
    df_jd['year'] = pd.to_datetime(df_jd['post_date']).dt.year
    
    # Đếm số lượng JD theo năm và ngành
    jd_by_year_category = df_jd.groupby(['year', 'category']).size().unstack(fill_value=0)
    
    print("\nSố lượng JD theo năm và ngành:")
    print(jd_by_year_category)
    
    # Vẽ biểu đồ
    jd_by_year_category.plot(kind='bar', stacked=True, figsize=(12, 6))
    plt.xlabel('Năm', fontsize=12)
    plt.ylabel('Số lượng JD', fontsize=12)
    plt.title('Phân bố Job Descriptions theo Năm và Ngành', fontsize=14, fontweight='bold')
    plt.legend(title='Ngành', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('../outputs/eda_jd_by_year_category.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Đã lưu biểu đồ tại: outputs/eda_jd_by_year_category.png")
else:
    print("Dataset không có cột 'post_date' hoặc 'category' để phân tích xu hướng")

In [ ]:
# Phân tích từ khóa kỹ năng phổ biến (dựa trên tần suất từ)
import re
from collections import Counter

def extract_keywords(text, min_length=3):
    """Trích xuất các từ khóa có độ dài >= min_length"""
    if pd.isna(text):
        return []
    # Chuyển về lowercase và tách từ
    words = re.findall(r'\b[a-zA-Z]{3,}\b', str(text).lower())
    return words

# Lấy mẫu 1000 JD để phân tích (để nhanh)
sample_jd = df_jd.sample(min(1000, len(df_jd)), random_state=42)

# Trích xuất tất cả từ khóa
all_keywords = []
for text in sample_jd['description']:
    all_keywords.extend(extract_keywords(text))

# Đếm tần suất
keyword_freq = Counter(all_keywords)

# Lọc bỏ các từ phổ biến không phải kỹ năng (stopwords cơ bản)
stopwords = {'the', 'and', 'for', 'with', 'you', 'are', 'our', 'will', 'can', 'has', 'this', 'that', 'from', 'your'}
tech_keywords = {k: v for k, v in keyword_freq.items() if k not in stopwords}

# Lấy top 30 kỹ năng
top_30_skills = sorted(tech_keywords.items(), key=lambda x: x[1], reverse=True)[:30]

print("=" * 60)
print("TOP 30 TỪ KHÓA KỸ NĂNG PHỔ BIẾN NHẤT TRONG JD")
print("=" * 60)
for i, (skill, count) in enumerate(top_30_skills, 1):
    print(f"{i:2d}. {skill:20s} - {count:4d} lần xuất hiện")

# Vẽ biểu đồ
skills_df = pd.DataFrame(top_30_skills, columns=['Skill', 'Count'])
plt.figure(figsize=(12, 8))
plt.barh(skills_df['Skill'][::-1], skills_df['Count'][::-1], color='coral')
plt.xlabel('Số lần xuất hiện', fontsize=12)
plt.ylabel('Kỹ năng', fontsize=12)
plt.title('Top 30 Từ khóa Kỹ năng Phổ biến trong Job Descriptions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/eda_top_skills.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Đã lưu biểu đồ tại: outputs/eda_top_skills.png")

In [ ]:
# Lưu dữ liệu đã lọc và làm sạch
print("=" * 60)
print("ĐANG LƯU DỮ LIỆU ĐÃ XỬ LÝ")
print("=" * 60)

# Tạo thư mục outputs nếu chưa có
Path('../outputs').mkdir(exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)

# Lưu dữ liệu
df_jd.to_csv('../data/processed/jd_filtered_2021_2026.csv', index=False)
df_cv.to_csv('../data/processed/cv_filtered_2021_2026.csv', index=False)

print(f"✓ Đã lưu JD: {len(df_jd)} bản ghi")
print(f"✓ Đã lưu CV: {len(df_cv)} bản ghi")
print(f"✓ Đường dẫn: data/processed/")

In [ ]:
# Tổng kết EDA
print("=" * 60)
print("TỔNG KẾT EDA TUẦN 1")
print("=" * 60)

print(f"""
✓ Dữ liệu đã được lọc trong khoảng 2021-2026:
  - Job Descriptions: {len(df_jd):,} bản ghi
  - Resumes/CV: {len(df_cv):,} bản ghi

✓ Phân tích độ dài văn bản:
  - JD trung bình: {df_jd['word_count'].mean():.0f} từ
  - CV trung bình: {df_cv['word_count'].mean():.0f} từ
  - JD dài gấp {df_jd['word_count'].mean() / df_cv['word_count'].mean():.1f} lần CV
  => Cần chiến lược xử lý bất đối xứng khi embedding

✓ Từ khóa kỹ năng phổ biến:
  - Top skills: {', '.join([s[0] for s in top_30_skills[:10]])}
  => Sẽ dùng để xây dựng tính năng "CV Health Check"

✓ Các file đã lưu:
  - data/processed/jd_filtered_2021_2026.csv
  - data/processed/cv_filtered_2021_2026.csv
  - outputs/eda_word_count_distribution.png
  - outputs/eda_top_skills.png

=> TUẦN 1 HOÀN THÀNH! Chuyển sang Tuần 2: Tiền xử lý & NER với spaCy
""")